<a href="https://colab.research.google.com/github/KiruthikaVM2005/My-project/blob/main/smartagri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 68.0 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from PIL import Image

st.set_page_config(page_title="Agri-Smart Pro", layout="centered")

st.markdown("""
<style>
.stApp {
    background-color: #f4fff4;
}

/* Title */
h1 {
    color: #1B5E20;
    text-align: center;
    font-weight: bold;
}

/* Section headers */
h2, h3 {
    color: #2E7D32;
}

/* Buttons */
.stButton>button {
    background-color: #2E7D32 !important;
    color: white !important;
    border-radius: 10px;
    height: 45px;
    font-weight: bold;
}

/* Input labels */
label {
    color: #1B5E20 !important;
    font-weight: 600;
}
</style>
""", unsafe_allow_html=True)

@st.cache_resource
def train_model():
    df = pd.read_csv('Crop_recommendation.csv')
    X = df[['N','P','K','temperature','humidity','ph','rainfall']]
    y = df['label']
    model = RandomForestClassifier(n_estimators=100)
    model.fit(X, y)
    return model, df

model, df = train_model()
all_crops = sorted(df['label'].unique())
region_crops = {
    "Tamil Nadu": ["rice", "banana", "coconut", "maize"],
    "Kerala": ["coconut", "banana", "rice"],
    "Punjab": ["wheat", "maize", "rice"],
    "Maharashtra": ["cotton", "maize"]
}

np.random.seed(42)
profit_data = {
    crop: np.random.randint(15000, 40000) for crop in all_crops
}

def predict_disease(image):
    img = np.array(image.resize((128,128))) / 255.0
    green = img[:,:,1].mean()
    red = img[:,:,0].mean()

    if red > 0.5:
        return "Leaf Blight", "Apply fungicide"
    elif green < 0.3:
        return "Powdery Mildew", "Use sulfur spray"
    else:
        return "Healthy", "No treatment needed"

if "page" not in st.session_state:
    st.session_state.page = "Home"

if st.session_state.page == "Home":

    st.title("🌿 Agri-Smart Pro AI")
    st.markdown("### Choose a Feature")

    col1, col2 = st.columns(2)

    with col1:
        if st.button("🌾 Crop Recommendation"):
            st.session_state.page = "Crop"

    with col2:
        if st.button("🔍 Disease Detection"):
            st.session_state.page = "Disease"

elif st.session_state.page == "Crop":

    st.title("🌾 Crop Recommendation")

    if st.button("⬅️ Back"):
        st.session_state.page = "Home"

    region = st.selectbox("🌍 Select Region", list(region_crops.keys()))

    st.markdown("### Enter Soil & Climate Details")

    col1, col2 = st.columns(2)

    with col1:
        n = st.number_input("Nitrogen (N)", 0, 150, 50)
        p = st.number_input("Phosphorus (P)", 0, 150, 50)
        k = st.number_input("Potassium (K)", 0, 150, 50)
        ph = st.number_input("pH", 0.0, 14.0, 6.5)

    with col2:
        temp = st.number_input("Temperature (°C)", 0.0, 50.0, 25.0)
        hum = st.number_input("Humidity (%)", 0.0, 100.0, 60.0)
        rain = st.number_input("Rainfall (mm)", 0.0, 500.0, 100.0)

    if st.button("Predict Crop"):

        probs = model.predict_proba([[n,p,k,temp,hum,ph,rain]])[0]
        crops = model.classes_

        top = np.argsort(probs)[-3:][::-1]

        st.divider()
        st.subheader("🌾 Top Crop Recommendations")

        for i in top:
            crop = crops[i]
            prob = probs[i]

            st.write(f"🌱 {crop} - {prob*100:.0f}%")
            st.progress(float(prob))

        st.divider()
        st.subheader("🌍 Region Insight")

        preferred = region_crops.get(region, [])

        for i in top:
            crop = crops[i]

            if crop in preferred:
                st.success(f"{crop} is highly suitable for {region}")
            else:
                st.info(f"{crop} can grow but not ideal for {region}")

        st.divider()
        st.subheader("💰 Profit Estimation")

        for i in top:
            crop = crops[i]
            st.write(f"{crop} → ₹{profit_data[crop]}")

        if ph < 5.5:
            st.warning("Soil is acidic")
        elif ph > 7.5:
            st.warning("Soil is alkaline")
        else:
            st.success("Soil pH is optimal")

elif st.session_state.page == "Disease":

    st.title("🔍 Disease Detection")

    if st.button("⬅️ Back"):
        st.session_state.page = "Home"

    file = st.file_uploader("Upload Leaf Image", type=["jpg","png","jpeg"])

    if file:
        image = Image.open(file)
        st.image(image, width=220)

        if st.button("Scan Disease"):
            disease, solution = predict_disease(image)

            st.error(f"🦠 Disease: {disease}")
            st.success(f"💊 Solution: {solution}")

Writing app.py


In [ ]:
!npm install -g localtunnel
import os
os.system("nohup streamlit run app.py &")
import requests
print("Your Password is:", requests.get('https://localtunnel.me/whitelist').text)

!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹Your Password is: {"id":"whitelist","port":32195,"max_conn_count":2,"url":"https://whitelist.loca.lt"}
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙your url is: https://cold-gifts-decide.loca.lt
